# ViSceT5 — Finetune + Ablation · nhánh `exp/ablation-drop-ocr` · Colab A100

**Cấu hình chính thức (ViSceT5):** CLIP ViT (đóng băng, **không** nhận câu hỏi) + **SceSpaVis** + **OCR-Aug**. Không pretrain, không visual search, không MRA.

| Tên | QA-ViT (câu hỏi vào CLIP) | SceSpaVis | OCR-Aug | Vai trò |
|---|---|---|---|---|
| `base` | ✗ | ✗ (OCR text-only) | ✗ | baseline |
| `qavit` | ✓ | ✗ | ✗ | ablation: thêm câu hỏi vào CLIP |
| `scespavis` | ✗ | ✓ | ✗ | ablation: thêm SceSpaVis |
| `ocraug` | ✗ | ✗ | ✓ | ablation: thêm OCR-Aug |
| `full` | ✗ | ✓ | ✓ | **ViSceT5 — kết quả chính** |
| `full-qavit` | ✓ | ✓ | ✓ | ablation: ViSceT5 + câu hỏi vào CLIP |

Mọi cấu hình: 5 epoch, batch 4×2, LR 3e-5, chọn model theo EM trên val, beam 4. Chạy lần lượt từ trên xuống; chọn cấu hình ở cell **Cấu hình**.

## 1. Clone đúng nhánh

In [ ]:
import os, sys
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")
BRANCH = "exp/ablation-drop-ocr"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin
!git checkout {BRANCH}
!git reset --hard origin/{BRANCH}
!git log --oneline -3
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Bảo đảm đã có bản sửa OCR-Aug ("<pad>" không bị "sửa" thành "ipad")
assert "_specials" in open("data/collator.py", encoding="utf-8").read(), "Thiếu bản sửa OCR-Aug — kiểm tra lại nhánh!"
print("✅ Nhánh", BRANCH, "có bản sửa OCR-Aug.")

## 2. Cài môi trường (transformers 4.45.2)

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap

In [ ]:
import torch, transformers
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("Transformers", transformers.__version__)
assert transformers.__version__.startswith("4.45"), "Cần transformers 4.45.x"

## 3. Đăng nhập Hugging Face (để tự push checkpoint mỗi epoch và tự resume khi Colab ngắt)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token vào ô hiện ra.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()   # dán token vào ô hiện ra, rồi chạy tiếp cell Cấu hình
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 4. Cấu hình

In [ ]:
import os
from huggingface_hub import get_token, HfApi

# finetune.py chỉ đọc HF_TOKEN từ biến môi trường → lấy token đã đăng nhập đưa vào env
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

DATASET = "ViTextVQA"          # ViTextVQA | ViOCRVQA | OpenViVQA
RUN_TAG = "r1"                 # đổi khi muốn chạy lại từ đầu với repo HF mới

# tên: (QA-ViT nhận câu hỏi, SceSpaVis, OCR-Aug)
CONFIGS = {
    "base":       (False, False, False),
    "qavit":      (True,  False, False),
    "scespavis":  (False, True,  False),
    "ocraug":     (False, False, True),
    "full":       (False, True,  True),    # ViSceT5 — KẾT QUẢ CHÍNH
    "full-qavit": (True,  True,  True),
}
RUN = ["full"]                 # cấu hình chạy lần này, vd ["full", "base", "scespavis"]

EPOCHS, BS, ACC, LR = 5, 4, 2, 3e-5
DATA_OUT = "./output/finetune_data"     # nơi finetune.py lưu/đọc CSV của dataset
TEST_GT_JSON = ""                       # (tuỳ chọn) đường dẫn JSON đáp án test để chấm offline

def out_dir(name):  return f"./output/abl_{DATASET}_{name}"
def hf_repo(name):  return f"{HF_USER}/ViSceT5-abl-{DATASET.lower()}-{name}-{RUN_TAG}"

for n in RUN:
    assert n in CONFIGS, n
    q, s, a = CONFIGS[n]
    print(f"{n:11s} QA-ViT={q!s:5} SceSpaVis={s!s:5} OCR-Aug={a!s:5} → {out_dir(n)} | {hf_repo(n)}")

## 5. Finetune (từ đầu, không pretrain)
`finetune.py` tự tải và chuẩn bị dữ liệu nếu chưa có, đẩy từng checkpoint lên repo HF của cấu hình đó và tự resume khi chạy lại. Xong mỗi cấu hình, model tốt nhất (theo EM val) được lưu ở `out_dir` và upload lên repo.

In [ ]:
import gc, torch
from huggingface_hub import HfApi

for name in RUN:
    q, s, a = CONFIGS[name]
    od = out_dir(name)
    env = (f"OUTPUT_PATH={DATA_OUT} HF_REPO={hf_repo(name)} RESUME_FROM_HF=auto "
           f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")
    args = ["configs/finetune.yaml",
            "--dataset_name", DATASET,
            "--ablation_use_vs", "False",
            "--ablation_use_qaclip", str(q),
            "--ablation_use_ocr", str(s),
            "--ablation_use_ocr_aug", str(a),
            "--num_train_epochs", str(EPOCHS),
            "--per_device_train_batch_size", str(BS),
            "--per_device_eval_batch_size", str(BS),
            "--gradient_accumulation_steps", str(ACC),
            "--learning_rate", str(LR),
            "--bf16", "True",
            "--output_dir", od,
            "--logging_dir", od + "/logs"]
    print("=" * 100, f"\n▶ {name}: {env} python training/finetune.py {' '.join(args)}\n" + "=" * 100)
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    get_ipython().system(env + " python training/finetune.py " + " ".join(args))

    # Upload model tốt nhất (finetune.py chỉ tự push checkpoint-*)
    if os.environ.get("HF_TOKEN") and os.path.exists(os.path.join(od, "model.safetensors")):
        try:
            api = HfApi(token=os.environ["HF_TOKEN"])
            api.create_repo(hf_repo(name), repo_type="model", exist_ok=True, private=True)
            api.upload_folder(folder_path=od, repo_id=hf_repo(name), repo_type="model",
                              ignore_patterns=["checkpoint-*/**", "*optimizer.pt", "*rng_state*", "*scheduler.pt"])
            print("☁️ Đã upload model tốt nhất →", hf_repo(name))
        except Exception as e:
            print("⚠️ Upload lỗi:", e)

## 6. Dự đoán dev + test và chấm điểm
`predict.py` sinh câu trả lời (beam 4) cho dev (in EM/F1) và test (`submission_*.csv`). Nếu có file đáp án test (`TEST_GT_JSON`), `eval_submission.py` chấm EM, F1, BLEU, CIDEr.

In [ ]:
import json, os
for name in RUN:
    od = out_dir(name)
    if not os.path.exists(os.path.join(od, "model.safetensors")):
        print(f"⚠️ {name}: chưa có model trong {od} — bỏ qua"); continue
    get_ipython().system(f"OUTPUT_PATH={DATA_OUT} python -m training.predict --split both --ckpt_dir {od} "
                         f"--dataset {DATASET} --batch_size 8 --num_beams 4 --out_dir {od}")
    tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
    pred = os.path.join(od, f"submission_{tag}test.csv")
    if TEST_GT_JSON and os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
        get_ipython().system(f"python -m training.eval_submission --gt {TEST_GT_JSON} --pred {pred} "
                             f"--out {od}/eval_test.json")
    elif not TEST_GT_JSON:
        print(f"ℹ️ {name}: chưa đặt TEST_GT_JSON → chỉ sinh {pred}, chưa chấm test.")

## 7. Tổng hợp kết quả các cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for name, (q, s, a) in CONFIGS.items():
    od = out_dir(name)
    r = {"config": name, "QA-ViT": q, "SceSpaVis": s, "OCR-Aug": a}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        js = json.load(open(st[-1]))
        evs = [h for h in js.get("log_history", []) if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h["eval_em"])
            r.update({"val_EM": round(100 * best["eval_em"], 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    ev = os.path.join(od, "eval_test.json")
    if os.path.exists(ev):
        r.update({f"test_{k}": v for k, v in json.load(open(ev)).items() if not isinstance(v, (dict, list))})
    if len(r) > 4:
        rows.append(r)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")